# AGHC-S128 — 01 Quick Reproduction

Fastest end-to-end reproduction: environment check, encrypt/decrypt round trip with metadata, and headline statistical metrics. Everything calls the `aghc_s128` package — the notebook contains no algorithm code. Statistical metrics are evaluations, not a security proof (docs/SECURITY_SCOPE.md).

In [ ]:
# --- Setup (local / Jupyter) -------------------------------------------
# Option A: repository checked out locally
#   pip install -e .            (run once in the repo root, outside Jupyter)
# Option B: pip install from a released wheel / archive
#   %pip install aghc-s128==1.0.0
#
# This cell only configures paths; nothing is executed on import of the package.


In [ ]:
# --- Optional: Google Colab setup ---------------------------------------
# Run this cell ONLY on Google Colab. It installs the package from the
# repository; everything else in the notebook then uses the same code as a
# local installation.
#
# From GitHub:
# !git clone https://github.com/PLACEHOLDER/aghc-s128-reproducibility.git
# %cd aghc-s128-reproducibility
# !pip install -e .
#
# Or from an uploaded archive (Workspace sidebar -> upload), then:
# !pip install /content/aghc-s128-reproducibility
#
# Google Drive is NOT required. To use Drive anyway (optional):
# from google.colab import drive
# drive.mount('/content/drive')


## 1 — Validate the environment

In [ ]:
import aghc_s128
from aghc_s128.cli import main
print('aghc_s128 version:', aghc_s128.__version__)
main(['validate'])

## 2 — Quick reproduction on a sample file
Password here is a **public demo test vector**, never a real secret.

In [ ]:
from pathlib import Path
from aghc_s128 import encrypt_bytes, decrypt_bytes
from aghc_s128.io_utils import read_binary_file

sample = Path('data/sample/sample_text.txt')
if not sample.exists():
    sample = Path('sample_text.txt')  # fallback: upload your own

password = 'aghc-s128-demo'
plaintext = read_binary_file(sample)
enc = encrypt_bytes(plaintext, n_input=8, password=password)
dec = decrypt_bytes(enc.ciphertext, n_input=8, password=password,
                    trial_index=enc.trial_index)
print('size             :', plaintext.size)
print('effective N      :', enc.effective_n, ' (N = 2 x n)')
print('hill / residual  :', enc.hill_length, '/', enc.residual_length)
print('trial index      :', enc.trial_index)
print('round trip ok    :', dec.plaintext_sha3_256 == enc.plaintext_sha3_256)
print('ciphertext sha3  :', enc.ciphertext_sha3_256)

## 3 — Mode A metrics at a glance (same library as the CLI)

In [ ]:
from aghc_s128 import metrics
m = {
    'entropy_original':  metrics.shannon_entropy(plaintext),
    'entropy_encrypted': metrics.shannon_entropy(enc.ciphertext),
    'corr_original':     metrics.adjacent_byte_correlation(plaintext),
    'corr_encrypted':    metrics.adjacent_byte_correlation(enc.ciphertext),
    'npcr_percent':      metrics.npcr(plaintext, enc.ciphertext),
    'uaci_percent':      metrics.uaci(plaintext, enc.ciphertext),
    'bit_diff_percent':  metrics.bit_difference_ratio(plaintext, enc.ciphertext),
}
for key, value in m.items():
    print(f'{key:20s} {value:.4f}')